In [ ]:
import ROOT as r

class MyJet(r.TLorentzVector):
    def __init__(self, px=0, py=0, pz=0, e=0, btag=0.0, jetid=False):
        super().__init__(px, py, pz, e)
        self.btag = btag
        self.jetid = jetid

    def SetBTagDiscriminator(self, x):
        self.btag = x

    def GetBTagDiscriminator(self):
        return self.btag

    def IsBTagged(self, th=1.74):
        return self.btag > th

    def SetJetID(self, jetid):
        self.jetid = jetid

    def GetJetID(self):
        return self.jetid


In [ ]:
class MyElectron(r.TLorentzVector):
    def __init__(self, px=0, py=0, pz=0, e=0, iso=0.0, charge=0):
        super().__init__(px, py, pz, e)
        self.isolation = iso
        self.charge = charge

    def SetIsolation(self, x):
        self.isolation = x

    def GetIsolation(self):
        return self.isolation

    def SetCharge(self, q):
        self.charge = q

    def GetCharge(self):
        return self.charge

    def IsIsolated(self, relcut=1.0):
        return self.isolation < relcut


In [ ]:
class MyPhoton(r.TLorentzVector):
    def __init__(self, px=0, py=0, pz=0, e=0, iso=0.0):
        super().__init__(px, py, pz, e)
        self.isolation = iso

    def SetIsolation(self, x):
        self.isolation = x

    def GetIsolation(self):
        return self.isolation

    def IsIsolated(self, relcut=1.0):
        return self.isolation < relcut

In [ ]:
class MyMuon(r.TLorentzVector):
    def __init__(self, px=0, py=0, pz=0, e=0, iso=0.0, charge=0):
        super().__init__(px, py, pz, e)
        self.isolation = iso
        self.charge = charge

    def IsIsolated(self, relcut=0.1):
        if self.Pt() == 0:
            return False

In [ ]:
class Plotter:
    def __init__(self):
        self.data = []
        self.bg = []
        self.signal = []
        self.data_names = []
        self.bg_names = []
        self.signal_names = []
        self.N_histos = 0

    def SetData(self, v, n):
        self.data.append(v)
        self.data_names.append(n)
        self.N_histos = len(v)

    def ClearData(self):
        self.data.clear()
        self.data_names.clear()

    def AddBg(self, v, n):
        self.bg.append(v)
        self.bg_names.append(n)
        self.N_histos = len(v)

    def ClearBg(self):
        self.bg.clear()
        self.bg_names.clear()

    def AddSig(self, v, n):
        self.signal.append(v)
        self.signal_names.append(n)
        self.N_histos = len(v)

    def ClearSig(self):
        self.signal.clear()
        self.signal_names.clear()

    def Plot(self, filename="result.pdf"):
        r.gROOT.Reset()

        MyStyle = r.TStyle("MyStyle", "My Root Styles")
        MyStyle.SetStatColor(0)
        MyStyle.SetCanvasColor(0)
        MyStyle.SetPadColor(0)
        MyStyle.SetPadBorderMode(0)
        MyStyle.SetCanvasBorderMode(0)
        MyStyle.SetFrameBorderMode(0)
        MyStyle.SetOptStat(0)
        MyStyle.SetStatBorderSize(2)
        MyStyle.SetOptTitle(0)
        MyStyle.SetPadTickX(1)
        MyStyle.SetPadTickY(1)
        MyStyle.SetPadBorderSize(2)
        MyStyle.SetPalette(51, 0)
        MyStyle.SetPadBottomMargin(0.15)
        MyStyle.SetPadTopMargin(0.05)
        MyStyle.SetPadLeftMargin(0.15)
        MyStyle.SetPadRightMargin(0.25)
        MyStyle.SetTitleColor(1)
        MyStyle.SetTitleFillColor(0)
        MyStyle.SetTitleFontSize(0.05)
        MyStyle.SetTitleBorderSize(0)
        MyStyle.SetLineWidth(1)
        MyStyle.SetHistLineWidth(3)
        MyStyle.SetLegendBorderSize(0)
        MyStyle.SetNdivisions(502, "x")
        MyStyle.SetMarkerSize(0.8)
        MyStyle.SetTickLength(0.03)
        MyStyle.SetTitleOffset(1.5, "x")
        MyStyle.SetTitleOffset(1.5, "y")
        MyStyle.SetTitleOffset(1.0, "z")
        MyStyle.SetLabelSize(0.05, "x")
        MyStyle.SetLabelSize(0.05, "y")
        MyStyle.SetLabelSize(0.05, "z")
        MyStyle.SetLabelOffset(0.03, "x")
        MyStyle.SetLabelOffset(0.03, "y")
        MyStyle.SetLabelOffset(0.03, "z")
        MyStyle.SetTitleSize(0.05, "x")
        MyStyle.SetTitleSize(0.05, "y")
        MyStyle.SetTitleSize(0.05, "z")
        r.gROOT.SetStyle("MyStyle")

        DrawLog = True

        for i in range(self.N_histos):
            hs = None
            Nset = len(self.data) + len(self.bg) + len(self.signal)
            if Nset > 20:
                Nset = 20
            l = r.TLegend(0.76, 0.95 - 0.8 * Nset / 20.0, 1.0, 0.95)
            l.SetFillStyle(1001)
            l.SetFillColor(r.kWhite)
            l.SetLineColor(r.kWhite)
            l.SetLineWidth(2)

            # Background stack
            if len(self.bg) > 0:
                hs = r.THStack(f"hs_{i}", self.bg[0][i].GetName())
                for j, bkg_group in enumerate(self.bg):
                    color_index = [r.kRed, r.kOrange, r.kYellow, r.kGreen,
                                r.kCyan, r.kBlue, r.kMagenta, r.kGray,
                                r.kGray + 2, r.kBlack]
                    fill_color = color_index[j] if j < len(color_index) else r.kBlack
                    bkg_group[i].SetFillColor(fill_color)
                    hs.Add(bkg_group[i])
                    l.AddEntry(bkg_group[i], self.bg_names[j], "f")

            c = r.TCanvas(f"c{i}", f"Canvas {i}", 800, 600)
            c.SetLogy(DrawLog)

            plotname = ""

            if len(self.data) > 0:
                plotname = self.data[0][i].GetName()
                data_hist = self.data[0][i]
                data_hist.SetMaximum(5 * data_hist.GetMaximum())
                data_hist.GetXaxis().SetTitleOffset(1.3)
                data_hist.GetYaxis().SetTitleOffset(1.3)
                data_hist.GetYaxis().SetTitle("Events")
                data_hist.GetXaxis().SetNdivisions(505)
                data_hist.Draw("")
                l.AddEntry(data_hist, self.data_names[0], "p")
                if hs:
                    hs.Draw("histsame")
                data_hist.SetMarkerStyle(20)
                data_hist.Draw("psame")
                l.Draw("same")
            elif len(self.data) == 0 and len(self.bg) > 0:
                plotname = self.bg[0][i].GetName()
                hs.Draw("hist")
                hs.GetXaxis().SetTitleOffset(1.3)
                hs.GetXaxis().SetNdivisions(505)
                hs.GetYaxis().SetTitleOffset(1.3)
                hs.GetYaxis().SetTitle("Events")
                if len(self.bg) > 0:
                    hs.GetXaxis().SetTitle(self.bg[0][i].GetXaxis().GetTitle())
                l.Draw("same")

            # Save pages to PDF
            if i == 0 and self.N_histos > 1:
                c.Print(f"{filename}(")
            elif i > 0 and i == self.N_histos - 1:
                c.Print(f"{filename})")
            else:
                c.Print(filename)


'''
if __name__ == "__main__":
    # Open ROOT file
    f = r.TFile("files_for_python/data.root")
    tree = f.Get("events")

    # Histograms
    h_NMuon = r.TH1F("Number of Muons", "Number of isolated muons", 7, 0, 7)
    h_Mmumu = r.TH1F("Mass mumu", "Invariant di-muon mass", 60, 60, 120)
    h_NElectron = r.TH1F("Number of Electrons", "Number of isolated electrons", 7, 0, 7)

    MuonRelIsoCut = 0.1
    MuonPtCut = 25
    ElectronRelIsoCut = 1.0

    for i in range(min(10, tree.GetEntries())):
        tree.GetEntry(i)

        # Build muons
        Muons = [MyMuon(tree.Muon_Px[j], tree.Muon_Py[j], tree.Muon_Pz[j],
                        tree.Muon_E[j], tree.Muon_Iso[j], tree.Muon_Charge[j])
                for j in range(tree.NMuon)]
        iso_muons = [m for m in Muons if m.IsIsolated(MuonRelIsoCut)]
        h_NMuon.Fill(len(iso_muons), getattr(tree, "EventWeight", 1.0))

        # Di-muon mass
        if len(iso_muons) > 1:
            if iso_muons[0].Pt() > MuonPtCut:
                h_Mmumu.Fill((iso_muons[0] + iso_muons[1]).M(),
                            getattr(tree, "EventWeight", 1.0))

        # Build electrons
        Electrons = [MyElectron(tree.Electron_Px[j], tree.Electron_Py[j], tree.Electron_Pz[j],
                                tree.Electron_E[j], tree.Electron_Iso[j], tree.Electron_Charge[j])
                    for j in range(tree.NElectron)]
        iso_electrons = [e for e in Electrons if e.IsIsolated(ElectronRelIsoCut)]
        h_NElectron.Fill(len(iso_electrons), getattr(tree, "EventWeight", 1.0))

    # Example: draw muon multiplicity histogram
    c = r.TCanvas("c1", "Histograms", 800, 600)
    h_NMuon.Draw()
    c.Update()'''

'\nif __name__ == "__main__":\n    # Open ROOT file\n    f = r.TFile("files_for_python/data.root")\n    tree = f.Get("events")\n\n    # Histograms\n    h_NMuon = r.TH1F("Number of Muons", "Number of isolated muons", 7, 0, 7)\n    h_Mmumu = r.TH1F("Mass mumu", "Invariant di-muon mass", 60, 60, 120)\n    h_NElectron = r.TH1F("Number of Electrons", "Number of isolated electrons", 7, 0, 7)\n\n    MuonRelIsoCut = 0.1\n    MuonPtCut = 25\n    ElectronRelIsoCut = 1.0\n\n    for i in range(min(10, tree.GetEntries())):\n        tree.GetEntry(i)\n\n        # Build muons\n        Muons = [MyMuon(tree.Muon_Px[j], tree.Muon_Py[j], tree.Muon_Pz[j],\n                        tree.Muon_E[j], tree.Muon_Iso[j], tree.Muon_Charge[j])\n                 for j in range(tree.NMuon)]\n        iso_muons = [m for m in Muons if m.IsIsolated(MuonRelIsoCut)]\n        h_NMuon.Fill(len(iso_muons), getattr(tree, "EventWeight", 1.0))\n\n        # Di-muon mass\n        if len(iso_muons) > 1:\n            if iso_muons[0].

In [ ]:
from math import sqrt, pow
from array import array

class MyAnalysis:
    def __init__(self, tree):
        self.tree = tree
        self.Muons = []
        self.Jets = []
        self.Electrons = []
        self.met = None
        
        # Cuts and constants
        self.MuonRelIsoCut = 0.1
        self.MuonPtCut = 26.0
        self.SF_b = 0.95
        self.triggerIsoMu24 = True
        
        # Event weights and counters
        self.EventWeight = 1.0
        self.NPrimaryVertices = 0
        self.N_IsoMuon = 0
        self.muon1 = None
        self.muon2 = None
        self.SelectedEvents = 0
        self.SelectedEvents_triggered = 0
        self.GeneratedEvents = 0
        self.IsSelected = False
        
        self.BookHistograms()
    
    def BookHistograms(self):
        # Exercise 1
        self.h_NMuon = r.TH1F("h_NMuon", "Number of isolated muons", 10, 0, 10)
        self.h_Mmumu = r.TH1F("h_Mmumu", "Invariant di-muon mass", 100, 0, 200)
        
        # Exercise 2
        self.h_nPV = r.TH1F("h_nPV", "Number of Primary Vertices", 50, 0, 50)
        self.h_Jet1_Pt = r.TH1F("h_Jet1_Pt", "Jet1 Pt", 100, 0, 300)
        self.h_Jet1_Eta = r.TH1F("h_Jet1_Eta", "Jet1 Eta", 50, -5, 5)
        self.h_Jet2_Pt = r.TH1F("h_Jet2_Pt", "Jet2 Pt", 100, 0, 300)
        self.h_Jet2_Eta = r.TH1F("h_Jet2_Eta", "Jet2 Eta", 50, -5, 5)
        self.h_Jet3_Pt = r.TH1F("h_Jet3_Pt", "Jet3 Pt", 100, 0, 300)
        self.h_Jet3_Eta = r.TH1F("h_Jet3_Eta", "Jet3 Eta", 50, -5, 5)
        self.h_BJet1_Pt = r.TH1F("h_BJet1_Pt", "BJet1 Pt", 100, 0, 300)
        self.h_BJet1_Eta = r.TH1F("h_BJet1_Eta", "BJet1 Eta", 50, -5, 5)
        self.h_BJet2_Pt = r.TH1F("h_BJet2_Pt", "BJet2 Pt", 100, 0, 300)
        self.h_BJet2_Eta = r.TH1F("h_BJet2_Eta", "BJet2 Eta", 50, -5, 5)
        self.h_NBJet = r.TH1F("h_NBJet", "Number of BJets", 10, 0, 10)
        self.h_NJet = r.TH1F("h_NJet", "Number of Jets", 10, 0, 10)
        self.h_Muon1_Pt = r.TH1F("h_Muon1_Pt", "Muon1 Pt", 100, 0, 300)
        self.h_Muon1_Eta = r.TH1F("h_Muon1_Eta", "Muon1 Eta", 50, -5, 5)
        self.h_Muon1_Iso = r.TH1F("h_Muon1_Iso", "Muon1 Iso", 50, 0, 1)
        self.h_Muon2_Pt = r.TH1F("h_Muon2_Pt", "Muon2 Pt", 100, 0, 300)
        self.h_Muon2_Eta = r.TH1F("h_Muon2_Eta", "Muon2 Eta", 50, -5, 5)
        self.h_Muon2_Iso = r.TH1F("h_Muon2_Iso", "Muon2 Iso", 50, 0, 1)
        self.h_Electron1_Pt = r.TH1F("h_Electron1_Pt", "Electron1 Pt", 100, 0, 300)
        self.h_Electron1_Eta = r.TH1F("h_Electron1_Eta", "Electron1 Eta", 50, -5, 5)
        self.h_NElectron = r.TH1F("h_NElectron", "Number of isolated electrons", 10, 0, 10)
        self.h_MET = r.TH1F("h_MET", "Missing transverse energy", 100, 0, 500)
        
        # Exercise 3
        self.h_selectedEvents_Muon1_Pt = r.TH1F("h_selectedEvents_Muon1_Pt", "Selected Muon1 Pt", 100, 0, 300)
        self.h_selectedEvents_triggered_Muon1_Pt = r.TH1F("h_selectedEvents_triggered_Muon1_Pt", "Selected triggered Muon1 Pt", 100, 0, 300)
        
        # Exercise 4
        self.h_Mbqqb_mc = r.TH1F("h_Mbqqb_mc", "Hadronic top MC", 100, 0, 500)
        self.h_Mbln_mc = r.TH1F("h_Mbln_mc", "Leptonic top MC", 100, 0, 500)
        self.h_Mbqqb_reco = r.TH1F("h_Mbqqb_reco", "Hadronic top reco", 100, 0, 500)
        self.h_Mbln_reco = r.TH1F("h_Mbln_reco", "Leptonic top reco", 100, 0, 500)
    
    def ProcessEvent(self):
        # exercise 1
        N_IsoMuon = 0
        self.muon1 = None
        self.muon2 = None
        for mu in self.Muons:
            if mu.IsIsolated(self.MuonRelIsoCut):
                N_IsoMuon += 1
                if N_IsoMuon == 1:
                    self.muon1 = mu
                elif N_IsoMuon == 2:
                    self.muon2 = mu
        self.h_NMuon.Fill(N_IsoMuon, self.EventWeight)
        if N_IsoMuon > 1 and self.triggerIsoMu24:
            if self.muon1.Pt() > self.MuonPtCut:
                self.h_Mmumu.Fill((self.muon1 + self.muon2).M(), self.EventWeight)
        
        # exercise 2
        N_BJet = sum(1 for j in self.Jets if j.IsBTagged())
        N_Jet = 0
        N_BJet_tmp = 0
        if self.triggerIsoMu24 and N_IsoMuon > 0 and self.muon1.Pt() > self.MuonPtCut:
            self.h_nPV.Fill(self.NPrimaryVertices, self.EventWeight)
            N_Jet = 0
            N_BJet_tmp = 0
            for j in self.Jets:
                if not j.GetJetID():
                    continue
                N_Jet += 1
                # Fill jets
                if N_Jet == 1:
                    self.h_Jet1_Pt.Fill(j.Pt(), self.EventWeight)
                    self.h_Jet1_Eta.Fill(j.Eta(), self.EventWeight)
                elif N_Jet == 2:
                    self.h_Jet2_Pt.Fill(j.Pt(), self.EventWeight)
                    self.h_Jet2_Eta.Fill(j.Eta(), self.EventWeight)
                elif N_Jet == 3:
                    self.h_Jet3_Pt.Fill(j.Pt(), self.EventWeight)
                    self.h_Jet3_Eta.Fill(j.Eta(), self.EventWeight)
                if j.IsBTagged():
                    N_BJet_tmp += 1
                    if N_BJet_tmp == 1:
                        self.h_BJet1_Pt.Fill(j.Pt(), self.EventWeight*self.SF_b)
                        self.h_BJet1_Eta.Fill(j.Eta(), self.EventWeight*self.SF_b)
                    elif N_BJet_tmp == 2:
                        self.h_BJet2_Pt.Fill(j.Pt(), self.EventWeight*self.SF_b*self.SF_b)
                        self.h_BJet2_Eta.Fill(j.Eta(), self.EventWeight*self.SF_b*self.SF_b)
            self.h_NBJet.Fill(N_BJet, self.EventWeight*pow(self.SF_b,N_BJet))
            self.h_NJet.Fill(N_Jet, self.EventWeight)
            
            # Isolated Muons
            N_IsoMuon_counter = 0
            for mu in self.Muons:
                if mu.IsIsolated(self.MuonRelIsoCut):
                    N_IsoMuon_counter += 1
                    if N_IsoMuon_counter == 1:
                        self.h_Muon1_Pt.Fill(mu.Pt(), self.EventWeight)
                        self.h_Muon1_Eta.Fill(mu.Eta(), self.EventWeight)
                        self.h_Muon1_Iso.Fill(mu.GetIsolation(), self.EventWeight)
                    elif N_IsoMuon_counter == 2:
                        self.h_Muon2_Pt.Fill(mu.Pt(), self.EventWeight)
                        self.h_Muon2_Eta.Fill(mu.Eta(), self.EventWeight)
                        self.h_Muon2_Iso.Fill(mu.GetIsolation(), self.EventWeight)
            
            # Isolated Electrons
            N_IsoElectron_counter = 0
            for el in self.Electrons:
                if el.GetIsolation()/el.Pt() < self.MuonRelIsoCut:
                    N_IsoElectron_counter += 1
                    if N_IsoElectron_counter == 1:
                        self.h_Electron1_Pt.Fill(el.Pt(), self.EventWeight)
                        self.h_Electron1_Eta.Fill(el.Eta(), self.EventWeight)
            self.h_NElectron.Fill(N_IsoElectron_counter, self.EventWeight)
            self.h_MET.Fill(self.met.Pt(), self.EventWeight)
        
        # exercise 3
        self.GeneratedEvents += self.EventWeight
        if N_IsoMuon == 1 and self.muon1.Pt() > self.MuonPtCut:
            NBJet = sum(1 for j in self.Jets if j.IsBTagged())
            if NBJet > 1:
                self.SelectedEvents += self.EventWeight*self.SF_b*self.SF_b
                if self.triggerIsoMu24:
                    self.SelectedEvents_triggered += self.EventWeight*self.SF_b*self.SF_b
                    self.IsSelected = True
            # Fill histograms
            if len(self.Muons) == 1:
                if self.Muons[0].IsIsolated(self.MuonRelIsoCut):
                    self.h_selectedEvents_Muon1_Pt.Fill(self.Muons[0].Pt(), self.EventWeight)
                    if self.triggerIsoMu24:
                        self.h_selectedEvents_triggered_Muon1_Pt.Fill(self.Muons[0].Pt(), self.EventWeight)
        
        # exercise 4
        # MC truth
        # hadB, hadWq, hadWqb, lepB, lepWl, lepWn must be defined in your framework
        # self.h_Mbqqb_mc.Fill((hadB + hadWq + hadWqb).M(), self.EventWeight)
        # self.h_Mbln_mc.Fill((lepB + lepWl + lepWn).M(), self.EventWeight)
        # Fit can be called in Python too: self.h_Mbqqb_mc.Fit("gaus")
        
        if self.IsSelected:
            # Hadronic top reco
            for i, it in enumerate(self.Jets):
                if it.IsBTagged():
                    for j, jt in enumerate(self.Jets):
                        if jt != it and not jt.IsBTagged():
                            for k, kt in enumerate(self.Jets):
                                if kt != it and kt != jt and not kt.IsBTagged():
                                    W_mass = (jt + kt).M()
                                    if 70 < W_mass < 95:
                                        self.h_Mbqqb_reco.Fill((it + jt + kt).M(), self.EventWeight)
            
            # Leptonic top reco
            for it in self.Jets:
                if it.IsBTagged():
                    px, py = self.met.Px(), self.met.Py()
                    mW = 80.379
                    mu = self.muon1
                    A = mW**2 / 2 + (mu.Px()*px + mu.Py()*py)
                    F = 2*A*mu.Pz()/(mu.Px()**2 + mu.Py()**2)
                    G = (mu.E()**2 * self.met.Pt()**2 - A**2)/(mu.Px()**2 + mu.Py()**2)
                    D = mu.E()**2 * (A**2 - self.met.Pt()**2 * (mu.E()**2 - mu.Pz()**2))
                    if D >= 0:
                        pz = F + sqrt(F**2)
                        E = sqrt(px**2 + py**2 + pz**2)
                        neutrino1 = r.TLorentzVector(px, py, pz, E)
                        self.h_Mbln_reco.Fill((it + mu + neutrino1).M(), self.EventWeight)
                    if D > 0:
                        pz = F - sqrt(F**2)
                        E = sqrt(px**2 + py**2 + pz**2)
                        neutrino2 = r.TLorentzVector(px, py, pz, E)
                        self.h_Mbln_reco.Fill((it + mu + neutrino2).M(), self.EventWeight)
    
    def DrawHistograms(self):
        # Draw all histograms
        import os
        save_folder = "plots"
        if not os.path.exists(save_folder):
            os.makedirs(save_folder)
        for attr in dir(self):
            if attr.startswith("h_"):
                h = getattr(self, attr)
                c = r.TCanvas(attr+"_c", attr+"_c", 800, 600)
                h.Draw()
                c.Update()
                c.SaveAs(f"{save_folder}/{attr}.png")


In [ ]:
# MyAnalysis.py
'''
import ROOT as r
from math import sqrt, pow
import sys

class MyAnalysis:
    def __init__(self, tree=None):
        # Event counters / weights / scale factors
        self.TotalEvents = 0
        self.GeneratedEvents = 0.0
        self.SelectedEvents = 0.0
        self.SelectedEvents_triggered = 0.0
        self.EventWeight = 1.0
        self.weight_factor = 1.0
        self.SF_b = 1.0
        self.triggerIsoMu24 = False

        # ROOT tree (set with set_tree or passed at construction)
        self.tree = tree

        # Collections (filled by BuildEvent)
        self.Muons = []
        self.Electrons = []
        self.Jets = []
        self.Photons = []
        self.met = r.TLorentzVector()

        # MC truth vectors (filled by BuildEvent if branches exist)
        self.hadB = r.TLorentzVector()
        self.lepB = r.TLorentzVector()
        self.hadWq = r.TLorentzVector()
        self.hadWqb = r.TLorentzVector()
        self.lepWl = r.TLorentzVector()
        self.lepWn = r.TLorentzVector()

        # Simple per-event bookkeeping used in Process
        self.NPrimaryVertices = 0
        self.NMuon = 0
        self.NElectron = 0
        self.NJet = 0
        self.NPhoton = 0

        # "cached" selection variables (kept similar to original code)
        self.N_IsoMuon = 0
        self.muon1 = None
        self.muon2 = None

        # Histograms: will be created in BookHistograms() / SlaveBegin
        self.hists_booked = False


    def set_tree(self, tree):
        self.tree = tree

    def BookHistograms(self):
        # Muons
        self.h_NMuon = r.TH1F("h_NMuon", "Number of isolated muons", 7, 0, 7)
        self.h_Mmumu = r.TH1F("h_Mmumu", "Invariant di-muon mass", 120, 40, 160)

        # Jets & b-jets
        self.h_nPV = r.TH1F("h_nPV", "N Primary Vertices", 50, 0, 50)
        self.h_Jet1_Pt = r.TH1F("h_Jet1_Pt", "Jet1 pT", 50, 0, 500)
        self.h_Jet1_Eta = r.TH1F("h_Jet1_Eta", "Jet1 eta", 50, -5, 5)
        self.h_Jet2_Pt = r.TH1F("h_Jet2_Pt", "Jet2 pT", 50, 0, 500)
        self.h_Jet2_Eta = r.TH1F("h_Jet2_Eta", "Jet2 eta", 50, -5, 5)
        self.h_Jet3_Pt = r.TH1F("h_Jet3_Pt", "Jet3 pT", 50, 0, 500)
        self.h_Jet3_Eta = r.TH1F("h_Jet3_Eta", "Jet3 eta", 50, -5, 5)

        self.h_BJet1_Pt = r.TH1F("h_BJet1_Pt", "BJet1 pT", 50, 0, 500)
        self.h_BJet1_Eta = r.TH1F("h_BJet1_Eta", "BJet1 eta", 50, -5, 5)
        self.h_BJet2_Pt = r.TH1F("h_BJet2_Pt", "BJet2 pT", 50, 0, 500)
        self.h_BJet2_Eta = r.TH1F("h_BJet2_Eta", "BJet2 eta", 50, -5, 5)

        self.h_NBJet = r.TH1F("h_NBJet", "Number of b-jets", 10, 0, 10)
        self.h_NJet = r.TH1F("h_NJet", "Number of jets", 10, 0, 10)

        # Muon kinematics
        self.h_Muon1_Pt = r.TH1F("h_Muon1_Pt", "Muon1 pT", 50, 0, 500)
        self.h_Muon1_Eta = r.TH1F("h_Muon1_Eta", "Muon1 eta", 50, -5, 5)
        self.h_Muon1_Iso = r.TH1F("h_Muon1_Iso", "Muon1 iso", 50, 0, 10)
        self.h_Muon2_Pt = r.TH1F("h_Muon2_Pt", "Muon2 pT", 50, 0, 500)
        self.h_Muon2_Eta = r.TH1F("h_Muon2_Eta", "Muon2 eta", 50, -5, 5)
        self.h_Muon2_Iso = r.TH1F("h_Muon2_Iso", "Muon2 iso", 50, 0, 10)

        # Electrons
        self.h_Electron1_Pt = r.TH1F("h_Electron1_Pt", "Electron1 pT", 50, 0, 500)
        self.h_Electron1_Eta = r.TH1F("h_Electron1_Eta", "Electron1 eta", 50, -5, 5)
        self.h_NElectron = r.TH1F("h_NElectron", "Number of isolated electrons", 7, 0, 7)

        # MET
        self.h_MET = r.TH1F("h_MET", "Missing ET", 60, 0, 600)

        # Selected/triggered muon histograms
        self.h_selectedEvents_Muon1_Pt = r.TH1F("h_selectedEvents_Muon1_Pt", "Selected Muon1 pT", 50, 0, 500)
        self.h_selectedEvents_triggered_Muon1_Pt = r.TH1F("h_selectedEvents_triggered_Muon1_Pt", "Triggered selected Muon1 pT", 50, 0, 500)

        # Top reconstruction — MC truth and reco
        self.h_Mbqqb_mc = r.TH1F("h_Mbqqb_mc", "MC hadronic top mass", 100, 100, 260)
        self.h_Mbln_mc = r.TH1F("h_Mbln_mc", "MC leptonic top mass", 100, 100, 260)
        self.h_Mbqqb_reco = r.TH1F("h_Mbqqb_reco", "Reco hadronic top mass", 100, 100, 260)
        self.h_Mbln_reco = r.TH1F("h_Mbln_reco", "Reco leptonic top mass", 100, 100, 260)

        # Flag
        self.hists_booked = True

    def GetEntry(self, entry):
        if self.tree is None:
            raise RuntimeError("No tree attached. Call set_tree(tree) before running.")
        
        self.tree.GetEntry(entry)

        # Copy scalar branches used in analysis (guard with hasattr)
        # Scalars
        for name in ("EventWeight", "NPrimaryVertices", "NMuon", "NElectron", "NJet", "NPhoton",
                    "MChadronicBottom_px", "MChadronicBottom_py", "MChadronicBottom_pz",
                    "MCleptonicBottom_px", "MCleptonicBottom_py", "MCleptonicBottom_pz",
                    "MChadronicWDecayQuark_px", "MChadronicWDecayQuark_py", "MChadronicWDecayQuark_pz",
                    "MChadronicWDecayQuarkBar_px", "MChadronicWDecayQuarkBar_py", "MChadronicWDecayQuarkBar_pz",
                    "MClepton_px", "MClepton_py", "MClepton_pz",
                    "MCneutrino_px", "MCneutrino_py", "MCneutrino_pz",
                    "MET_px", "MET_py",
                    "SF_b", "weight_factor", "triggerIsoMu24"):
            if hasattr(self.tree, name):
                setattr(self, name, getattr(self.tree, name))

        # Branch arrays (vectors) - keep references to tree arrays for indexing
        arr_names = [
            ("Muon_Px", "Muon_Py", "Muon_Pz", "Muon_E", "Muon_Iso", "Muon_Charge"),
            ("Electron_Px", "Electron_Py", "Electron_Pz", "Electron_E", "Electron_Iso", "Electron_Charge"),
            ("Photon_Px", "Photon_Py", "Photon_Pz", "Photon_E", "Photon_Iso"),
            ("Jet_Px", "Jet_Py", "Jet_Pz", "Jet_E", "Jet_btag", "Jet_ID")
        ]
        # Attach arrays if present
        if hasattr(self.tree, "NMuon"):
            self.NMuon = int(self.tree.NMuon)
        else:
            self.NMuon = getattr(self, "NMuon", 0)
        if hasattr(self.tree, "NElectron"):
            self.NElectron = int(self.tree.NElectron)
        else:
            self.NElectron = getattr(self, "NElectron", 0)
        if hasattr(self.tree, "NJet"):
            self.NJet = int(self.tree.NJet)
        else:
            self.NJet = getattr(self, "NJet", 0)
        if hasattr(self.tree, "NPhoton"):
            self.NPhoton = int(self.tree.NPhoton)
        else:
            self.NPhoton = getattr(self, "NPhoton", 0)

        # We will access the tree branch arrays directly in BuildEvent (e.g. self.tree.Muon_Px[i])
        # so no further copies are made here.

        # Set default EventWeight
        if hasattr(self.tree, "EventWeight"):
            self.EventWeight = getattr(self.tree, "EventWeight")
        else:
            self.EventWeight = getattr(self, "EventWeight", 1.0)

        # Update SF_b and weight_factor if in tree
        if hasattr(self.tree, "SF_b"):
            self.SF_b = getattr(self.tree, "SF_b")
        if hasattr(self.tree, "weight_factor"):
            self.weight_factor = getattr(self.tree, "weight_factor")
        if hasattr(self.tree, "triggerIsoMu24"):
            self.triggerIsoMu24 = bool(getattr(self.tree, "triggerIsoMu24"))

        # Build TLorentzVectors for MC truth and MET if branches exist
        if hasattr(self.tree, "MChadronicBottom_px"):
            self.hadB.SetXYZM(getattr(self, "MChadronicBottom_px"),
                            getattr(self, "MChadronicBottom_py"),
                            getattr(self, "MChadronicBottom_pz"),
                            4.8)
        if hasattr(self.tree, "MCleptonicBottom_px"):
            self.lepB.SetXYZM(getattr(self, "MCleptonicBottom_px"),
                            getattr(self, "MCleptonicBottom_py"),
                            getattr(self, "MCleptonicBottom_pz"),
                            4.8)
        if hasattr(self.tree, "MChadronicWDecayQuark_px"):
            self.hadWq.SetXYZM(getattr(self, "MChadronicWDecayQuark_px"),
                            getattr(self, "MChadronicWDecayQuark_py"),
                            getattr(self, "MChadronicWDecayQuark_pz"),
                            0.0)
        if hasattr(self.tree, "MChadronicWDecayQuarkBar_px"):
            self.hadWqb.SetXYZM(getattr(self, "MChadronicWDecayQuarkBar_px"),
                                getattr(self, "MChadronicWDecayQuarkBar_py"),
                                getattr(self, "MChadronicWDecayQuarkBar_pz"),
                                0.0)
        if hasattr(self.tree, "MClepton_px"):
            self.lepWl.SetXYZM(getattr(self, "MClepton_px"),
                            getattr(self, "MClepton_py"),
                            getattr(self, "MClepton_pz"),
                            0.0)
        if hasattr(self.tree, "MCneutrino_px"):
            self.lepWn.SetXYZM(getattr(self, "MCneutrino_px"),
                            getattr(self, "MCneutrino_py"),
                            getattr(self, "MCneutrino_pz"),
                            0.0)
        if hasattr(self.tree, "MET_px"):
            self.met.SetXYZM(getattr(self, "MET_px"), getattr(self, "MET_py"), 0.0, 0.0)

    def BuildEvent(self):
        # Build Muons
        self.Muons = []
        if hasattr(self.tree, "Muon_Px"):
            for i in range(self.NMuon):
                px = self.tree.Muon_Px[i]
                py = self.tree.Muon_Py[i]
                pz = self.tree.Muon_Pz[i]
                e = self.tree.Muon_E[i]
                iso = self.tree.Muon_Iso[i] if hasattr(self.tree, "Muon_Iso") else 0.0
                charge = int(self.tree.Muon_Charge[i]) if hasattr(self.tree, "Muon_Charge") else 0
                mu = r.TLorentzVector(px, py, pz, e)
                # We want to use your MyMuon class; if present, use it, otherwise use TLorentzVector wrapper
                try:
                    mymu = MyMuon(px, py, pz, e, iso, charge)
                except NameError:
                    # fallback to TLorentzVector-like object with .Pt(), .Eta(), etc.
                    mymu = mu
                    mymu.isolation = iso
                    mymu.charge = charge
                self.Muons.append(mymu)

        # Build Electrons
        self.Electrons = []
        if hasattr(self.tree, "Electron_Px"):
            for i in range(self.NElectron):
                px = self.tree.Electron_Px[i]
                py = self.tree.Electron_Py[i]
                pz = self.tree.Electron_Pz[i]
                e = self.tree.Electron_E[i]
                iso = self.tree.Electron_Iso[i] if hasattr(self.tree, "Electron_Iso") else 0.0
                charge = int(self.tree.Electron_Charge[i]) if hasattr(self.tree, "Electron_Charge") else 0
                try:
                    el = MyElectron(px, py, pz, e, iso, charge)
                except NameError:
                    el = r.TLorentzVector(px, py, pz, e)
                    el.isolation = iso
                    el.charge = charge
                self.Electrons.append(el)

        # Build Photons
        self.Photons = []
        if hasattr(self.tree, "Photon_Px"):
            for i in range(self.NPhoton):
                px = self.tree.Photon_Px[i]
                py = self.tree.Photon_Py[i]
                pz = self.tree.Photon_Pz[i]
                e = self.tree.Photon_E[i]
                iso = self.tree.Photon_Iso[i] if hasattr(self.tree, "Photon_Iso") else 0.0
                try:
                    ph = MyPhoton(px, py, pz, e, iso)
                except NameError:
                    ph = r.TLorentzVector(px, py, pz, e)
                    ph.isolation = iso
                self.Photons.append(ph)

        # Build Jets
        self.Jets = []
        if hasattr(self.tree, "Jet_Px"):
            for i in range(self.NJet):
                px = self.tree.Jet_Px[i]
                py = self.tree.Jet_Py[i]
                pz = self.tree.Jet_Pz[i]
                e = self.tree.Jet_E[i]
                btag = self.tree.Jet_btag[i] if hasattr(self.tree, "Jet_btag") else 0.0
                jetid = bool(self.tree.Jet_ID[i]) if hasattr(self.tree, "Jet_ID") else True
                try:
                    jet = MyJet(px, py, pz, e, btag, jetid)
                except NameError:
                    jet = r.TLorentzVector(px, py, pz, e)
                    jet.btag = btag
                    jet.jetid = jetid
                    def IsBTagged_local(th=1.74, _jet=jet):
                        return getattr(_jet, "btag", 0.0) > th
                    def GetJetID_local(_jet=jet):
                        return getattr(_jet, "jetid", True)
                    jet.IsBTagged = IsBTagged_local
                    jet.GetJetID = GetJetID_local
                self.Jets.append(jet)

        # EventWeight scaling
        self.EventWeight *= getattr(self, "weight_factor", 1.0)

        self.N_IsoMuon = 0
        self.muon1 = None
        self.muon2 = None
        for mu in self.Muons:
            
            try:
                is_iso = mu.IsIsolated(0.10)
            except AttributeError:
                is_iso = (getattr(mu, "isolation", 0.0) / (mu.Pt() if mu.Pt() != 0 else 1.0)) < 0.10
            if is_iso:
                self.N_IsoMuon += 1
                if self.N_IsoMuon == 1:
                    self.muon1 = mu
                elif self.N_IsoMuon == 2:
                    self.muon2 = mu

    def Process(self, entry):
        # Mirror the original C++ Process()
        self.TotalEvents += 1
        self.GetEntry(entry)

        if self.TotalEvents % 10000 == 0:
            print("Next event ----->", self.TotalEvents)

        self.BuildEvent()

        MuonPtCut = 25.0
        MuonRelIsoCut = 0.1

        

        # Exercise 1: Invariant Di-Muon mass
        N_IsoMuon = 0
        muon1 = None
        muon2 = None
        for mu in self.Muons:
            try:
                iso = mu.IsIsolated(MuonRelIsoCut)
            except Exception:
                iso = (getattr(mu, "isolation", 0.0) / (mu.Pt() if mu.Pt() != 0 else 1.0)) < MuonRelIsoCut
            if iso:
                N_IsoMuon += 1
                if N_IsoMuon == 1:
                    muon1 = mu
                elif N_IsoMuon == 2:
                    muon2 = mu

        # Fill N muon hist
        if not self.hists_booked:
            self.BookHistograms()
        self.h_NMuon.Fill(N_IsoMuon, self.EventWeight)

        if N_IsoMuon > 1 and self.triggerIsoMu24:
            if muon1 is not None and muon1.Pt() > MuonPtCut:
                self.h_Mmumu.Fill((muon1 + muon2).M(), self.EventWeight)

        c = r.TCanvas("c1", "Histograms", 800, 600)
        self.h_Mmumu.Draw()
        c.Update()

        # Exercise 2: jets, b-jets, muon/electron kinematics, MET
        N_BJet = sum(1 for jet in self.Jets if hasattr(jet, "IsBTagged") and jet.IsBTagged())
        N_Jet = 0
        N_BJet_tmp = 0

        if self.triggerIsoMu24 and N_IsoMuon > 0 and muon1 is not None and muon1.Pt() > MuonPtCut:
            self.h_nPV.Fill(getattr(self, "NPrimaryVertices", 0), self.EventWeight)

            for jet in self.Jets:
                # require JetID
                jetid = True
                try:
                    jetid = jet.GetJetID()
                except Exception:
                    jetid = getattr(jet, "jetid", True)
                if not jetid:
                    continue

                N_Jet += 1
                if N_Jet == 1:
                    self.h_Jet1_Pt.Fill(jet.Pt(), self.EventWeight)
                    self.h_Jet1_Eta.Fill(jet.Eta(), self.EventWeight)
                elif N_Jet == 2:
                    self.h_Jet2_Pt.Fill(jet.Pt(), self.EventWeight)
                    self.h_Jet2_Eta.Fill(jet.Eta(), self.EventWeight)
                elif N_Jet == 3:
                    self.h_Jet3_Pt.Fill(jet.Pt(), self.EventWeight)
                    self.h_Jet3_Eta.Fill(jet.Eta(), self.EventWeight)

                # b-jet handling
                is_b = False
                try:
                    is_b = jet.IsBTagged()
                except Exception:
                    is_b = getattr(jet, "btag", 0.0) > 1.74
                if is_b:
                    N_BJet_tmp += 1
                    if N_BJet_tmp == 1:
                        self.h_BJet1_Pt.Fill(jet.Pt(), self.EventWeight * self.SF_b)
                        self.h_BJet1_Eta.Fill(jet.Eta(), self.EventWeight * self.SF_b)
                    elif N_BJet_tmp == 2:
                        self.h_BJet2_Pt.Fill(jet.Pt(), self.EventWeight * self.SF_b * self.SF_b)
                        self.h_BJet2_Eta.Fill(jet.Eta(), self.EventWeight * self.SF_b * self.SF_b)

            self.h_NBJet.Fill(N_BJet, self.EventWeight * pow(self.SF_b, N_BJet))
            self.h_NJet.Fill(N_Jet, self.EventWeight)

            # Isolated muon plots
            N_IsoMuon_counter = 0
            for mu in self.Muons:
                try:
                    isolated = mu.IsIsolated(MuonRelIsoCut)
                except Exception:
                    isolated = (getattr(mu, "isolation", 0.0) / (mu.Pt() if mu.Pt() != 0 else 1.0)) < MuonRelIsoCut
                if isolated:
                    N_IsoMuon_counter += 1
                    if N_IsoMuon_counter == 1:
                        self.h_Muon1_Pt.Fill(mu.Pt(), self.EventWeight)
                        self.h_Muon1_Eta.Fill(mu.Eta(), self.EventWeight)
                        try:
                            self.h_Muon1_Iso.Fill(mu.GetIsolation(), self.EventWeight)
                        except Exception:
                            self.h_Muon1_Iso.Fill(getattr(mu, "isolation", 0.0), self.EventWeight)
                    elif N_IsoMuon_counter == 2:
                        self.h_Muon2_Pt.Fill(mu.Pt(), self.EventWeight)
                        self.h_Muon2_Eta.Fill(mu.Eta(), self.EventWeight)
                        try:
                            self.h_Muon2_Iso.Fill(mu.GetIsolation(), self.EventWeight)
                        except Exception:
                            self.h_Muon2_Iso.Fill(getattr(mu, "isolation", 0.0), self.EventWeight)

            # Isolated electrons
            N_IsoElectron_counter = 0
            for el in self.Electrons:
                try:
                    isol = el.GetIsolation()
                except Exception:
                    isol = getattr(el, "isolation", 0.0)
                # protect against Pt==0
                ptel = el.Pt() if el.Pt() != 0 else 1e-6
                if (isol / ptel) < MuonRelIsoCut:
                    N_IsoElectron_counter += 1
                    if N_IsoElectron_counter == 1:
                        self.h_Electron1_Pt.Fill(el.Pt(), self.EventWeight)
                        self.h_Electron1_Eta.Fill(el.Eta(), self.EventWeight)
            self.h_NElectron.Fill(N_IsoElectron_counter, self.EventWeight)

            # MET
            self.h_MET.Fill(self.met.Pt(), self.EventWeight)

        # Exercise 3: trigger checks and acceptance efficiency
        if len(self.Muons) == 1:
            mu = self.Muons[0]
            try:
                isolated = mu.IsIsolated(MuonRelIsoCut)
            except Exception:
                isolated = (getattr(mu, "isolation", 0.0) / (mu.Pt() if mu.Pt() != 0 else 1.0)) < MuonRelIsoCut
            if isolated:
                self.h_selectedEvents_Muon1_Pt.Fill(mu.Pt(), self.EventWeight)
                if self.triggerIsoMu24:
                    self.h_selectedEvents_triggered_Muon1_Pt.Fill(mu.Pt(), self.EventWeight)

        # Acceptance counters
        self.GeneratedEvents += self.EventWeight
        IsSelected = False
        if N_IsoMuon == 1 and muon1 is not None and muon1.Pt() > MuonPtCut:
            NBJet = sum(1 for j in self.Jets if hasattr(j, "IsBTagged") and j.IsBTagged())
            if NBJet > 1:
                self.SelectedEvents += self.EventWeight * self.SF_b * self.SF_b
                if self.triggerIsoMu24:
                    self.SelectedEvents_triggered += self.EventWeight * self.SF_b * self.SF_b
                    IsSelected = True

        # Exercise 4: top mass MC truth and reconstruction
        if not self.hists_booked:
            self.BookHistograms()

        # MC-level top masses (if hadB/hadWq etc. set earlier in GetEntry)
        try:
            self.h_Mbqqb_mc.Fill((self.hadB + self.hadWq + self.hadWqb).M(), self.EventWeight)
            self.h_Mbqqb_mc.Fit("gaus")
        except Exception:
            # ignore if MC truth not available
            pass
        try:
            self.h_Mbln_mc.Fill((self.lepB + self.lepWl + self.lepWn).M(), self.EventWeight)
            self.h_Mbln_mc.Fit("gaus")
        except Exception:
            pass

        if IsSelected:
            # Hadronic channel reconstruction: look for b-tagged jet + two non-b jets with W mass window
            for i, it in enumerate(self.Jets):
                try:
                    if not it.IsBTagged():
                        continue
                except Exception:
                    if not getattr(it, "btag", 0.0) > 1.74:
                        continue
                for j, jt in enumerate(self.Jets):
                    if j <= i:
                        continue
                    try:
                        if jt.IsBTagged():
                            continue
                    except Exception:
                        if getattr(jt, "btag", 0.0) > 1.74:
                            continue
                    for k, kt in enumerate(self.Jets):
                        if k <= j:
                            continue
                        try:
                            if kt.IsBTagged():
                                continue
                        except Exception:
                            if getattr(kt, "btag", 0.0) > 1.74:
                                continue
                        Wmass = (jt + kt).M()
                        if 70 < Wmass < 95:
                            self.h_Mbqqb_reco.Fill((it + jt + kt).M(), self.EventWeight)

            # Leptonic channel: reconstruct neutrino pz from W constraint
            if muon1 is None:
                # ensure we have a muon1 to use
                pass
            else:
                for it in self.Jets:
                    try:
                        if not it.IsBTagged():
                            continue
                    except Exception:
                        if not (getattr(it, "btag", 0.0) > 1.74):
                            continue

                    px = self.met.Px()
                    py = self.met.Py()
                    mW = 80.379

                    # Compute A, F, G, D (following the original algebra)
                    A = (mW ** 2) / 2.0 + (muon1.Px() * px + muon1.Py() * py)
                    denom = (muon1.Px() ** 2 + muon1.Py() ** 2)
                    # protect divide by zero
                    if denom == 0:
                        continue
                    F = (2 * A * muon1.Pz()) / denom
                    # G is not used later in code but keep computed for completeness
                    try:
                        G = ((muon1.E() ** 2) * (self.met.Pt() ** 2) - A ** 2) / denom
                    except Exception:
                        G = 0.0
                    D = muon1.E() ** 2 * (A ** 2 - (self.met.Pt() ** 2) * (muon1.E() ** 2 - muon1.Pz() ** 2))

                    # Solve for neutrino pz (two solutions if discriminant positive)
                    if D >= 0:
                        pz = F + sqrt(F ** 2)
                        E = sqrt(px ** 2 + py ** 2 + pz ** 2)
                        neutrino1 = r.TLorentzVector(px, py, pz, E)
                        self.h_Mbln_reco.Fill((it + muon1 + neutrino1).M(), self.EventWeight)

                    if D > 0:
                        pz = F - sqrt(F ** 2)
                        E = sqrt(px ** 2 + py ** 2 + pz ** 2)
                        neutrino2 = r.TLorentzVector(px, py, pz, E)
                        self.h_Mbln_reco.Fill((it + muon1 + neutrino2).M(), self.EventWeight)

        return True

    def Run(self, max_entries=None):
        if self.tree is None:
            raise RuntimeError("No tree set. Attach a TTree via set_tree().")
        nentries = int(self.tree.GetEntries())
        if max_entries is not None:
            nentries = min(nentries, int(max_entries))
        for i in range(nentries):
            ok = self.Process(i)
            if not ok:
                break
        # After loop
        self.Terminate()

    def SlaveTerminate(self):
        # In PROOF style workflows this would be run on slaves.
        pass

    def Terminate(self):
        # Run at the end of the job: draw some default plots (optional)
        # Example: draw the muon multiplicity
        try:
            c = r.TCanvas("c_end", "End Plots", 800, 600)
            self.h_NMuon.Draw()
            c.Update()
            # Save canvas if desired:
            # c.Print("h_NMuon.pdf")
        except Exception:
            pass

'''

'\nimport ROOT as r\nfrom math import sqrt, pow\nimport sys\n\n# If your classes are in same file/module, import is unnecessary; otherwise:\n# from myobjects import MyMuon, MyElectron, MyJet, MyPhoton\n# (Assumes MyMuon etc. are available in the same namespace.)\n\nclass MyAnalysis:\n    def __init__(self, tree=None):\n        # Event counters / weights / scale factors\n        self.TotalEvents = 0\n        self.GeneratedEvents = 0.0\n        self.SelectedEvents = 0.0\n        self.SelectedEvents_triggered = 0.0\n        self.EventWeight = 1.0\n        self.weight_factor = 1.0\n        self.SF_b = 1.0\n        self.triggerIsoMu24 = False\n\n        # ROOT tree (set with set_tree or passed at construction)\n        self.tree = tree\n\n        # Collections (filled by BuildEvent)\n        self.Muons = []\n        self.Electrons = []\n        self.Jets = []\n        self.Photons = []\n        self.met = r.TLorentzVector()\n\n        # MC truth vectors (filled by BuildEvent if branches exi